In [5]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from pathlib import Path

DATA_PATH = Path("../news_data/cleaned_news_for_model.parquet")
df_model = pd.read_parquet(DATA_PATH)

print(DATA_PATH.resolve())
print(df_model.shape)
display(df_model.head(3))

C:\Users\skuts\vibecode_projects\news_data\cleaned_news_for_model.parquet
(146619, 10)


,source,url,archive_date,published_at,title,text,category_raw,collected_at,text_len_chars,text_len_words
0,lenta.ru,https://lenta.ru/news/2025/01/01/auto/,2025-01-01,2025-01-01 00:00:00,"Рост акцизов на топливо, увеличение утильсбора...",В России с 1 января 2025 года вступает в силу ...,Россия,2026-04-24T08:54:39+00:00,10241,1699
1,lenta.ru,https://lenta.ru/news/2025/01/01/v-rossii-stal...,2025-01-01,2025-01-01 00:02:00,В России стало дороже развестись,В России кратно увеличился размер госпошлины з...,Россия,2026-04-24T08:54:39+00:00,2361,335
2,lenta.ru,https://lenta.ru/news/2025/01/01/v-rossii-s-1-...,2025-01-01,2025-01-01 01:23:00,В России с 1 января повысили штрафы за нарушен...,В России с 1 января повысили штрафы за превыше...,Россия,2026-04-24T08:54:39+00:00,2940,472


In [11]:
df = df_model[["title", "text", "category_raw"]].copy()
df = df.dropna(subset=["text", "category_raw"])
df = df[df["text"].str.len() > 0]
df["category_raw"].value_counts()

category_raw
Мир                45383
Россия             41352
Экономика          32112
Наука и техника    10168
Спорт               9869
Культура            7735
Name: count, dtype: int64

In [13]:
X = df["text"]
y = df["category_raw"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=3
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [15]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)


Accuracy: 0.9407652434865639
Macro F1: 0.9490410720340847
Weighted F1: 0.9406596909615853


In [16]:
print(classification_report(y_test, y_pred))

                 precision    recall  f1-score   support

       Культура       0.92      0.98      0.95      1547
            Мир       0.95      0.95      0.95      9077
Наука и техника       0.94      0.96      0.95      2034
         Россия       0.94      0.90      0.92      8270
          Спорт       0.99      0.99      0.99      1974
      Экономика       0.92      0.94      0.93      6422

       accuracy                           0.94     29324
      macro avg       0.94      0.95      0.95     29324
   weighted avg       0.94      0.94      0.94     29324



In [40]:
labels = model.classes_
cm = confusion_matrix(y_test, y_pred, labels = labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{x}" for x in labels],
    columns=[f"pred_{x}" for x in labels]
)
cm_df

,pred_Культура,pred_Мир,pred_Наука и техника,pred_Россия,pred_Спорт,pred_Экономика
true_Культура,1513,12,0,13,1,8
true_Мир,11,8663,33,231,2,137
true_Наука и техника,1,30,1953,33,0,17
true_Россия,60,281,55,7484,14,376
true_Спорт,4,4,0,3,1958,5
true_Экономика,47,159,27,166,7,6016


In [35]:

cm_row_norm = cm / cm.sum(axis=1, keepdims=True)

cm_row_norm_df = pd.DataFrame(
    cm_row_norm,
    index=[f"true_{x}" for x in labels],
    columns=[f"pred_{x}" for x in labels]
)

cm_row_norm_df.round(3)

,pred_Культура,pred_Мир,pred_Наука и техника,pred_Россия,pred_Спорт,pred_Экономика
true_Культура,0.978,0.008,0.000,0.008,0.001,0.005
true_Мир,0.001,0.954,0.004,0.025,0.000,0.015
true_Наука и техника,0.000,0.015,0.960,0.016,0.000,0.008
true_Россия,0.007,0.034,0.007,0.905,0.002,0.045
true_Спорт,0.002,0.002,0.000,0.002,0.992,0.003
true_Экономика,0.007,0.025,0.004,0.026,0.001,0.937


In [28]:
cm_col_norm = cm / cm.sum(axis=0, keepdims=True)

cm_col_norm_df = pd.DataFrame(
    cm_col_norm,
    index=[f"true_{x}" for x in labels],
    columns=[f"pred_{x}" for x in labels]
)

cm_col_norm_df.round(3)

,pred_Культура,pred_Мир,pred_Наука и техника,pred_Россия,pred_Спорт,pred_Экономика
true_Культура,0.925,0.001,0.000,0.002,0.001,0.001
true_Мир,0.007,0.947,0.016,0.029,0.001,0.021
true_Наука и техника,0.001,0.003,0.944,0.004,0.000,0.003
true_Россия,0.037,0.031,0.027,0.944,0.007,0.057
true_Спорт,0.002,0.000,0.000,0.000,0.988,0.001
true_Экономика,0.029,0.017,0.013,0.021,0.004,0.917


In [18]:
errors = pd.DataFrame({
    "text": X_test,
    "true": y_test,
    "pred": y_pred
})

errors = errors[errors["true"] != errors["pred"]]
errors.head(20)

,text,true,pred
42845,Правительство одобрило увеличение пособия по б...,Россия,Экономика
131084,"Минтранс сообщил, что ВСУ атаковали БЭКами рос...",Мир,Россия
79924,ДОМ.РФ профинансировал реконструкцию троллейбу...,Россия,Экономика
41823,Мединский: Россия будет ждать украинскую делег...,Россия,Мир
128187,Посол РФ во Франции Мешков: Получено 170 заяво...,Мир,Россия
10548,Генштаб заявил о наступлении ВС РФ почти на вс...,Россия,Наука и техника
43342,Стоимость проезда на городском транспорте Моск...,Россия,Экономика
34379,СФР: В связи с майскими праздниками пенсии рос...,Россия,Экономика
52837,Глава ФРС Пауэлл: Рост инфляции в США в ближай...,Мир,Экономика
60921,Москалькова: Украинские военнопленные чаще все...,Мир,Россия


In [41]:
errors_matrix = cm_df.copy()

for i in range(len(labels)):
    errors_matrix.iloc[i, i] = 0

top_errors = (
    errors_matrix
    .stack()
    .sort_values(ascending=False)
    .head(20)
)

top_errors

true_Россия           pred_Экономика          376
                      pred_Мир                281
true_Мир              pred_Россия             231
true_Экономика        pred_Россия             166
                      pred_Мир                159
true_Мир              pred_Экономика          137
true_Россия           pred_Культура            60
                      pred_Наука и техника     55
true_Экономика        pred_Культура            47
true_Мир              pred_Наука и техника     33
true_Наука и техника  pred_Россия              33
                      pred_Мир                 30
true_Экономика        pred_Наука и техника     27
true_Наука и техника  pred_Экономика           17
true_Россия           pred_Спорт               14
true_Культура         pred_Россия              13
                      pred_Мир                 12
true_Мир              pred_Культура            11
true_Культура         pred_Экономика            8
true_Экономика        pred_Спорт                7


In [54]:
error_rows = []

for i, true_label in enumerate(labels):
    for j, pred_label in enumerate(labels):
        if i != j:
            error_rows.append({
                "true": true_label,
                "pred": pred_label,
                "count": cm[i, j],
                "share_of_true_class": cm[i, j] / cm[i].sum(),
                "share_of_pred_class": cm[i, j] / cm[:, j].sum()
            })

error_analysis = pd.DataFrame(error_rows)

error_analysis.sort_values("count", ascending=False).head(30)

,true,pred,count,share_of_true_class,share_of_pred_class
19,Россия,Экономика,376,0.045466,0.057326
16,Россия,Мир,281,0.033978,0.030714
7,Мир,Россия,231,0.025449,0.029130
28,Экономика,Россия,166,0.025849,0.020933
26,Экономика,Мир,159,0.024759,0.017379
9,Мир,Экономика,137,0.015093,0.020887
15,Россия,Культура,60,0.007255,0.036675
17,Россия,Наука и техника,55,0.006651,0.026596
25,Экономика,Культура,47,0.007319,0.028729
12,Наука и техника,Россия,33,0.016224,0.004161
